# Ideal Portfolio Proposal

This notebook converts recommendations in `Analysis/docs/Full Research.md` into concrete proposed portfolios with target weights.

## Proposal A — Full Research Based
- Uses the original research-guided diversification approach.
- Emphasizes quality growth + defensive sectors + international + liquidity.

Proposal A is fully built/validated first before Proposal B begins.

### 1) Build Full Research Portfolio
This section builds Proposal A from research-based allocations.

Run the next code cell to create `full_research_df`.

In [15]:
import pandas as pd
from pathlib import Path
from io import StringIO

candidate_paths = [
    Path("full_research_proposal_weights.csv"),
    Path("Analysis/full_research_proposal_weights.csv"),
]

full_research_source_path = next((path for path in candidate_paths if path.exists()), None)
if full_research_source_path is None:
    raise FileNotFoundError(
        "Could not find full_research_proposal_weights.csv in current folder or Analysis/."
    )

full_research_df = pd.read_csv(full_research_source_path)
full_research_df["ticker"] = full_research_df["ticker"].astype(str).str.upper().str.strip()

seed_weights_csv = """Symbol,Target_Weight_Pct,Category
XLK,5.0,Growth Engines
VGT,5.0,Growth Engines
FTEC,5.0,Growth Engines
IYW,5.0,Growth Engines
SOXX,5.0,Growth Engines
XLF,3.0,Cyclical Leaders
VFH,3.0,Cyclical Leaders
IYF,3.0,Cyclical Leaders
KBWB,3.0,Cyclical Leaders
EUFN,3.0,Cyclical Leaders
XLI,2.5,Industrial Growth
VIS,2.5,Industrial Growth
FIDU,2.5,Industrial Growth
PPA,2.5,Industrial Growth
ITA,2.5,Industrial Growth
XLV,2.2,Health Core
VHT,2.2,Health Core
IHF,2.2,Health Core
IHI,2.2,Health Core
XHS,2.2,Health Core
XLE,2.0,Energy Upside
VDE,2.0,Energy Upside
IEO,2.0,Energy Upside
XOP,2.0,Energy Upside
OIH,2.0,Energy Upside
XLB,1.8,Materials/Mining
VAW,1.8,Materials/Mining
MXI,1.8,Materials/Mining
IYM,1.8,Materials/Mining
PICK,1.8,Materials/Mining
VNQ,1.5,Real Estate
SCHH,1.5,Real Estate
XLRE,1.5,Real Estate
IYR,1.5,Real Estate
RWR,1.5,Real Estate
XLU,1.0,Utilities
VPU,1.0,Utilities
FUTY,1.0,Utilities
IDU,1.0,Utilities
UTES,1.0,Utilities
XLP,1.0,Staples
VDC,1.0,Staples
FSTA,1.0,Staples
RHS,1.0,Staples
KXI,1.0,Staples
"""

seed_df = pd.read_csv(StringIO(seed_weights_csv))
seed_df = seed_df.rename(
    columns={
        "Symbol": "ticker",
        "Target_Weight_Pct": "seed_target_weight_pct",
        "Category": "category",
    }
)
seed_df["ticker"] = seed_df["ticker"].astype(str).str.upper().str.strip()

missing_in_research_csv = sorted(set(seed_df["ticker"]) - set(full_research_df["ticker"]))
if missing_in_research_csv:
    raise ValueError(
        f"Seed symbols not found in full research CSV: {missing_in_research_csv}"
    )

full_research_df = full_research_df.drop(columns=["seed_target_weight_pct"], errors="ignore")
full_research_df = full_research_df.drop(columns=["category"], errors="ignore")
full_research_df = full_research_df.merge(
    seed_df[["ticker", "seed_target_weight_pct", "category"]],
    on="ticker",
    how="left",
)

if full_research_df["seed_target_weight_pct"].isna().any():
    missing_seed = sorted(
        full_research_df.loc[full_research_df["seed_target_weight_pct"].isna(), "ticker"].unique().tolist()
    )
    raise ValueError(f"Missing seed weights for research tickers: {missing_seed}")

full_research_df["target_weight_pct"] = pd.to_numeric(
    full_research_df["seed_target_weight_pct"], errors="coerce"
).fillna(0.0)
full_research_df = full_research_df.drop(columns=["seed_target_weight_pct"] )

seed_total = float(full_research_df["target_weight_pct"].sum())
if seed_total <= 0:
    raise ValueError("Seed total weight must be greater than 0.")

if abs(seed_total - 100.0) > 1e-9:
    full_research_df["target_weight_pct"] = (
        full_research_df["target_weight_pct"] / seed_total * 100.0
    )

full_research_df["target_weight_pct"] = full_research_df["target_weight_pct"].round(4)

final_total = float(full_research_df["target_weight_pct"].sum())
print("Proposal A — Full Research Based (Seeded and Rebalanced)")
print(f"Source: {full_research_source_path}")
print(f"Positions: {len(full_research_df)}")
print(f"Seed total before normalization: {seed_total:.4f}%")
print(f"Total weight after normalization: {final_total:.4f}%")

display(
    full_research_df.sort_values("target_weight_pct", ascending=False)[
        [
            "ticker", "name", "sector", "category", "target_weight_pct", "role",
        ]
    ]
)

full_research_df

Proposal A — Full Research Based (Seeded and Rebalanced)
Source: full_research_proposal_weights.csv
Positions: 45
Seed total before normalization: 100.0000%
Total weight after normalization: 100.0000%


,ticker,name,sector,category,target_weight_pct,role
41,VGT,Vanguard Information Technology ETF,Technology,Growth Engines,5.0,Broad tech growth exposure
40,XLK,Technology Select Sector SPDR Fund,Technology,Growth Engines,5.0,US large-cap technology core
42,FTEC,Fidelity MSCI Information Technology Index ETF,Technology,Growth Engines,5.0,Low-cost technology diversification
43,IYW,iShares U.S. Technology ETF,Technology,Growth Engines,5.0,Liquid broad US technology exposure
44,SOXX,iShares Semiconductor ETF,Technology,Growth Engines,5.0,Semiconductor growth accelerator
26,VFH,Vanguard Financials ETF,Financials,Cyclical Leaders,3.0,Broad financial subsector diversification
27,IYF,iShares U.S. Financials ETF,Financials,Cyclical Leaders,3.0,Balanced large and mid financials
25,XLF,Financial Select Sector SPDR Fund,Financials,Cyclical Leaders,3.0,US financial sector anchor
29,EUFN,iShares MSCI Europe Financials ETF,Financials,Cyclical Leaders,3.0,European financial diversification
28,KBWB,Invesco KBW Bank ETF,Financials,Cyclical Leaders,3.0,Bank-heavy cyclical exposure


,proposal,pillar,ticker,name,asset_type,sector,style,region,strategy_bucket,target_weight_pct,role,target_weight_pct_seed,category
0,FullResearch,Research,XLV,Health Care Select Sector SPDR Fund,ETF,Healthcare,Defensive Growth,US,Sector ETF Basket,2.2,Healthcare anchor with diversified blue-chip e...,2.2,Health Core
1,FullResearch,Research,VHT,Vanguard Health Care ETF,ETF,Healthcare,Defensive Growth,US,Sector ETF Basket,2.2,Broader healthcare coverage with biotech tilt,2.2,Health Core
2,FullResearch,Research,IHF,iShares U.S. Healthcare Providers ETF,ETF,Healthcare,Provider Focus,US,Sector ETF Basket,2.2,Healthcare services and insurers exposure,2.2,Health Core
3,FullResearch,Research,IHI,iShares U.S. Medical Devices ETF,ETF,Healthcare,Innovation Tilt,US,Sector ETF Basket,2.2,Medical devices growth exposure,2.2,Health Core
4,FullResearch,Research,XHS,SPDR S&P Health Care Services ETF,ETF,Healthcare,Balanced Services,US,Sector ETF Basket,2.2,Balanced health services allocation,2.2,Health Core
5,FullResearch,Research,XLP,Consumer Staples Select Sector SPDR Fund,ETF,Consumer Staples,Defensive,US,Sector ETF Basket,1.0,Staples stability and low volatility anchor,1.0,Staples
6,FullResearch,Research,VDC,Vanguard Consumer Staples ETF,ETF,Consumer Staples,Defensive,US,Sector ETF Basket,1.0,Broad low-cost staples exposure,1.0,Staples
7,FullResearch,Research,FSTA,Fidelity MSCI Consumer Staples Index ETF,ETF,Consumer Staples,Defensive,US,Sector ETF Basket,1.0,Low-cost staples diversification,1.0,Staples
8,FullResearch,Research,RHS,Invesco S&P 500 Equal Weight Consumer Staples ETF,ETF,Consumer Staples,Equal Weight,US,Sector ETF Basket,1.0,Equal-weight staples diversification,1.0,Staples
9,FullResearch,Research,KXI,iShares Global Consumer Staples ETF,ETF,Consumer Staples,Global Defensive,Global,Sector ETF Basket,1.0,Global consumer staples diversification,1.0,Staples


### 2) Validate and Export Full Research
This section validates Proposal A allocations and exports the Full Research CSV.

Run this after building Proposal A.

In [16]:
def summarize_proposal(df: pd.DataFrame, proposal_name: str):
    total_weight = float(df["target_weight_pct"].sum())
    print(f"{proposal_name} total weight: {total_weight:.1f}%")

    if abs(total_weight - 100.0) < 1e-9:
        print("✅ Weights sum to 100%")
    else:
        print("⚠️ Weights do not sum to 100%")

    print("\nPillar Allocation:")
    display(df.groupby("pillar", as_index=False)["target_weight_pct"].sum().sort_values("target_weight_pct", ascending=False))

    print("\nAsset Type Allocation:")
    display(df.groupby("asset_type", as_index=False)["target_weight_pct"].sum().sort_values("target_weight_pct", ascending=False))

    print("\nTop Holdings:")
    display(df[["ticker", "name", "pillar", "target_weight_pct", "role"]].sort_values("target_weight_pct", ascending=False).head(10))


summarize_proposal(full_research_df, "Proposal A — Full Research Based")

full_research_path = "full_research_proposal_weights.csv"
full_research_df.to_csv(full_research_path, index=False)

print("\nSaved:")
print(f"- {full_research_path}")

Proposal A — Full Research Based total weight: 100.0%
✅ Weights sum to 100%

Pillar Allocation:


,pillar,target_weight_pct
0,Research,100.0



Asset Type Allocation:


,asset_type,target_weight_pct
0,ETF,100.0



Top Holdings:


,ticker,name,pillar,target_weight_pct,role
41,VGT,Vanguard Information Technology ETF,Research,5.0,Broad tech growth exposure
40,XLK,Technology Select Sector SPDR Fund,Research,5.0,US large-cap technology core
42,FTEC,Fidelity MSCI Information Technology Index ETF,Research,5.0,Low-cost technology diversification
43,IYW,iShares U.S. Technology ETF,Research,5.0,Liquid broad US technology exposure
44,SOXX,iShares Semiconductor ETF,Research,5.0,Semiconductor growth accelerator
26,VFH,Vanguard Financials ETF,Research,3.0,Broad financial subsector diversification
27,IYF,iShares U.S. Financials ETF,Research,3.0,Balanced large and mid financials
25,XLF,Financial Select Sector SPDR Fund,Research,3.0,US financial sector anchor
29,EUFN,iShares MSCI Europe Financials ETF,Research,3.0,European financial diversification
28,KBWB,Invesco KBW Bank ETF,Research,3.0,Bank-heavy cyclical exposure



Saved:
- full_research_proposal_weights.csv


## Proposal B — Fortress Portfolio (5 Pillars)
- Uses the 5-pillar framework you provided from the video:
  - Bedrock 45%
  - Fuel 20%
  - Shield 10%
  - Escape 15%
  - Rocket 10%

### 1) Build Fortress Portfolio
Run the next code cell to create `fortress_df`.

In [9]:
fortress_positions = [
    {"proposal": "Fortress", "pillar": "Bedrock", "ticker": "VTI", "name": "Vanguard Total Stock Market ETF", "asset_type": "ETF", "sector": "Broad Market", "style": "Core Index", "region": "US", "strategy_bucket": "Core ETF", "target_weight_pct": 20.0, "role": "US total market core"},
    {"proposal": "Fortress", "pillar": "Bedrock", "ticker": "VOO", "name": "Vanguard S&P 500 ETF", "asset_type": "ETF", "sector": "Broad Market", "style": "Core Index", "region": "US", "strategy_bucket": "Core ETF", "target_weight_pct": 15.0, "role": "US large-cap core"},
    {"proposal": "Fortress", "pillar": "Bedrock", "ticker": "VT", "name": "Vanguard Total World Stock ETF", "asset_type": "ETF", "sector": "Broad Market", "style": "Global Core", "region": "Global", "strategy_bucket": "Core ETF", "target_weight_pct": 7.0, "role": "Global equity anchor"},
    {"proposal": "Fortress", "pillar": "Bedrock", "ticker": "SPYM", "name": "SPDR Portfolio S&P 500 High Dividend ETF", "asset_type": "ETF", "sector": "Broad Market", "style": "Large Cap Blend", "region": "US", "strategy_bucket": "Core ETF", "target_weight_pct": 3.0, "role": "Low-cost S&P complement"},
    {"proposal": "Fortress", "pillar": "Fuel", "ticker": "SCHD", "name": "Schwab U.S. Dividend Equity ETF", "asset_type": "ETF", "sector": "Dividend Equity", "style": "Income Quality", "region": "US", "strategy_bucket": "Income ETF", "target_weight_pct": 8.0, "role": "Dividend stability"},
    {"proposal": "Fortress", "pillar": "Fuel", "ticker": "VIG", "name": "Vanguard Dividend Appreciation ETF", "asset_type": "ETF", "sector": "Dividend Equity", "style": "Dividend Growth", "region": "US", "strategy_bucket": "Income ETF", "target_weight_pct": 7.0, "role": "Dividend growth quality"},
    {"proposal": "Fortress", "pillar": "Fuel", "ticker": "DGRO", "name": "iShares Core Dividend Growth ETF", "asset_type": "ETF", "sector": "Dividend Equity", "style": "Dividend Growth", "region": "US", "strategy_bucket": "Income ETF", "target_weight_pct": 5.0, "role": "Broad dividend growth"},
    {"proposal": "Fortress", "pillar": "Shield", "ticker": "GLD", "name": "SPDR Gold Shares", "asset_type": "ETF", "sector": "Commodities", "style": "Gold", "region": "Global", "strategy_bucket": "Inflation Hedge", "target_weight_pct": 4.0, "role": "Liquid gold hedge"},
    {"proposal": "Fortress", "pillar": "Shield", "ticker": "IAU", "name": "iShares Gold Trust", "asset_type": "ETF", "sector": "Commodities", "style": "Gold", "region": "Global", "strategy_bucket": "Inflation Hedge", "target_weight_pct": 3.0, "role": "Low-cost gold hedge"},
    {"proposal": "Fortress", "pillar": "Shield", "ticker": "SGLN", "name": "abrdn Physical Gold Shares ETF", "asset_type": "ETF", "sector": "Commodities", "style": "Gold", "region": "Global", "strategy_bucket": "Inflation Hedge", "target_weight_pct": 3.0, "role": "Physical gold allocation"},
    {"proposal": "Fortress", "pillar": "Escape", "ticker": "VXUS", "name": "Vanguard Total International Stock ETF", "asset_type": "ETF", "sector": "International Equity", "style": "International Broad", "region": "Global ex-US", "strategy_bucket": "International ETF", "target_weight_pct": 6.0, "role": "Broad global diversification"},
    {"proposal": "Fortress", "pillar": "Escape", "ticker": "VEA", "name": "Vanguard FTSE Developed Markets ETF", "asset_type": "ETF", "sector": "International Equity", "style": "Developed Markets", "region": "Developed ex-US", "strategy_bucket": "International ETF", "target_weight_pct": 5.0, "role": "Developed market stability"},
    {"proposal": "Fortress", "pillar": "Escape", "ticker": "VWO", "name": "Vanguard FTSE Emerging Markets ETF", "asset_type": "ETF", "sector": "International Equity", "style": "Emerging Markets", "region": "Emerging Markets", "strategy_bucket": "International ETF", "target_weight_pct": 4.0, "role": "Emerging market growth"},
    {"proposal": "Fortress", "pillar": "Rocket", "ticker": "SMH", "name": "VanEck Semiconductor ETF", "asset_type": "ETF", "sector": "Technology", "style": "Semiconductors", "region": "Global", "strategy_bucket": "Growth ETF", "target_weight_pct": 4.0, "role": "High-conviction semiconductor growth"},
    {"proposal": "Fortress", "pillar": "Rocket", "ticker": "SOXX", "name": "iShares Semiconductor ETF", "asset_type": "ETF", "sector": "Technology", "style": "Semiconductors", "region": "US", "strategy_bucket": "Growth ETF", "target_weight_pct": 3.0, "role": "US chip ecosystem growth"},
    {"proposal": "Fortress", "pillar": "Rocket", "ticker": "QQQM", "name": "Invesco NASDAQ 100 ETF", "asset_type": "ETF", "sector": "Technology", "style": "Large Cap Growth", "region": "US", "strategy_bucket": "Growth ETF", "target_weight_pct": 3.0, "role": "Broad tech growth exposure"},
]

fortress_df = pd.DataFrame(fortress_positions)

print("Proposal B — Fortress (5 Pillars)")
display(fortress_df.sort_values(["pillar", "target_weight_pct"], ascending=[True, False]))

fortress_df

Proposal B — Fortress (5 Pillars)


,proposal,pillar,ticker,name,asset_type,sector,style,region,strategy_bucket,target_weight_pct,role
0,Fortress,Bedrock,VTI,Vanguard Total Stock Market ETF,ETF,Broad Market,Core Index,US,Core ETF,20.0,US total market core
1,Fortress,Bedrock,VOO,Vanguard S&P 500 ETF,ETF,Broad Market,Core Index,US,Core ETF,15.0,US large-cap core
2,Fortress,Bedrock,VT,Vanguard Total World Stock ETF,ETF,Broad Market,Global Core,Global,Core ETF,7.0,Global equity anchor
3,Fortress,Bedrock,SPYM,SPDR Portfolio S&P 500 High Dividend ETF,ETF,Broad Market,Large Cap Blend,US,Core ETF,3.0,Low-cost S&P complement
10,Fortress,Escape,VXUS,Vanguard Total International Stock ETF,ETF,International Equity,International Broad,Global ex-US,International ETF,6.0,Broad global diversification
11,Fortress,Escape,VEA,Vanguard FTSE Developed Markets ETF,ETF,International Equity,Developed Markets,Developed ex-US,International ETF,5.0,Developed market stability
12,Fortress,Escape,VWO,Vanguard FTSE Emerging Markets ETF,ETF,International Equity,Emerging Markets,Emerging Markets,International ETF,4.0,Emerging market growth
4,Fortress,Fuel,SCHD,Schwab U.S. Dividend Equity ETF,ETF,Dividend Equity,Income Quality,US,Income ETF,8.0,Dividend stability
5,Fortress,Fuel,VIG,Vanguard Dividend Appreciation ETF,ETF,Dividend Equity,Dividend Growth,US,Income ETF,7.0,Dividend growth quality
6,Fortress,Fuel,DGRO,iShares Core Dividend Growth ETF,ETF,Dividend Equity,Dividend Growth,US,Income ETF,5.0,Broad dividend growth


,proposal,pillar,ticker,name,asset_type,sector,style,region,strategy_bucket,target_weight_pct,role
0,Fortress,Bedrock,VTI,Vanguard Total Stock Market ETF,ETF,Broad Market,Core Index,US,Core ETF,20.0,US total market core
1,Fortress,Bedrock,VOO,Vanguard S&P 500 ETF,ETF,Broad Market,Core Index,US,Core ETF,15.0,US large-cap core
2,Fortress,Bedrock,VT,Vanguard Total World Stock ETF,ETF,Broad Market,Global Core,Global,Core ETF,7.0,Global equity anchor
3,Fortress,Bedrock,SPYM,SPDR Portfolio S&P 500 High Dividend ETF,ETF,Broad Market,Large Cap Blend,US,Core ETF,3.0,Low-cost S&P complement
4,Fortress,Fuel,SCHD,Schwab U.S. Dividend Equity ETF,ETF,Dividend Equity,Income Quality,US,Income ETF,8.0,Dividend stability
5,Fortress,Fuel,VIG,Vanguard Dividend Appreciation ETF,ETF,Dividend Equity,Dividend Growth,US,Income ETF,7.0,Dividend growth quality
6,Fortress,Fuel,DGRO,iShares Core Dividend Growth ETF,ETF,Dividend Equity,Dividend Growth,US,Income ETF,5.0,Broad dividend growth
7,Fortress,Shield,GLD,SPDR Gold Shares,ETF,Commodities,Gold,Global,Inflation Hedge,4.0,Liquid gold hedge
8,Fortress,Shield,IAU,iShares Gold Trust,ETF,Commodities,Gold,Global,Inflation Hedge,3.0,Low-cost gold hedge
9,Fortress,Shield,SGLN,abrdn Physical Gold Shares ETF,ETF,Commodities,Gold,Global,Inflation Hedge,3.0,Physical gold allocation


### 2) Validate and Export Fortress
This section validates Proposal B and exports the Fortress CSV.

It also exports the combined comparison CSV after both proposals exist.

In [10]:
def summarize_proposal(df: pd.DataFrame, proposal_name: str):
    total_weight = float(df["target_weight_pct"].sum())
    print(f"{proposal_name} total weight: {total_weight:.1f}%")

    if abs(total_weight - 100.0) < 1e-9:
        print("✅ Weights sum to 100%")
    else:
        print("⚠️ Weights do not sum to 100%")

    print("\nPillar Allocation:")
    display(df.groupby("pillar", as_index=False)["target_weight_pct"].sum().sort_values("target_weight_pct", ascending=False))

    print("\nAsset Type Allocation:")
    display(df.groupby("asset_type", as_index=False)["target_weight_pct"].sum().sort_values("target_weight_pct", ascending=False))

    print("\nTop Holdings:")
    display(df[["ticker", "name", "pillar", "target_weight_pct", "role"]].sort_values("target_weight_pct", ascending=False).head(10))


summarize_proposal(fortress_df, "Proposal B — Fortress (5 Pillars)")

fortress_path = "fortress_proposal_weights.csv"
fortress_df.to_csv(fortress_path, index=False)

portfolio_df = pd.concat([full_research_df, fortress_df], ignore_index=True)
comparison_path = "proposal_comparison_all_positions.csv"
portfolio_df.to_csv(comparison_path, index=False)

print("\nSaved:")
print(f"- {fortress_path}")
print(f"- {comparison_path}")

Proposal B — Fortress (5 Pillars) total weight: 100.0%
✅ Weights sum to 100%

Pillar Allocation:


,pillar,target_weight_pct
0,Bedrock,45.0
2,Fuel,20.0
1,Escape,15.0
3,Rocket,10.0
4,Shield,10.0



Asset Type Allocation:


,asset_type,target_weight_pct
0,ETF,100.0



Top Holdings:


,ticker,name,pillar,target_weight_pct,role
0,VTI,Vanguard Total Stock Market ETF,Bedrock,20.0,US total market core
1,VOO,Vanguard S&P 500 ETF,Bedrock,15.0,US large-cap core
4,SCHD,Schwab U.S. Dividend Equity ETF,Fuel,8.0,Dividend stability
2,VT,Vanguard Total World Stock ETF,Bedrock,7.0,Global equity anchor
5,VIG,Vanguard Dividend Appreciation ETF,Fuel,7.0,Dividend growth quality
10,VXUS,Vanguard Total International Stock ETF,Escape,6.0,Broad global diversification
11,VEA,Vanguard FTSE Developed Markets ETF,Escape,5.0,Developed market stability
6,DGRO,iShares Core Dividend Growth ETF,Fuel,5.0,Broad dividend growth
13,SMH,VanEck Semiconductor ETF,Rocket,4.0,High-conviction semiconductor growth
7,GLD,SPDR Gold Shares,Shield,4.0,Liquid gold hedge



Saved:
- fortress_proposal_weights.csv
- proposal_comparison_all_positions.csv


## Notes
- This is a proposal template derived from the research document, not personalized investment advice.
- You can tune weights by risk profile: growth tilt (increase NVDA/AMZN/META) or defensive tilt (increase JNJ/PG/XLU/CASH).
- Rebalance quarterly or after major valuation/fundamental changes.